# Native BioT5 Diverse Beam Local Review

This notebook evaluates `QizhiPei/biot5-plus-base-chebi20` with the active collection generator contract.
It uses the checkpoint-native tokenizer and SELFIES-first decoding flow, then compares greedy and diverse beam generation on a small ChEBI review slice.


## Notes

- This is a local review notebook, not a Kaggle notebook.
- It uses the native BioT5 checkpoint and tokenizer directly.
- The prompt contract is the original text2mol contract: description in, SELFIES out.
- The notebook decodes SELFIES first, then converts valid outputs to SMILES for similarity review.
- `filter_selfies(...)` is kept only as a diagnostic recovery fallback when the decoded text contains noisy extra characters.


In [1]:
import importlib.util

required_packages = ["selfies", "transformers", "sentencepiece", "huggingface_hub"]
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]

print("If you need a local install in this notebook, use:")
print("%pip install -U " + " ".join(required_packages))

if missing_packages:
    print("Missing packages:", missing_packages)
    print("Install them with %pip in this notebook, then restart the kernel.")
else:
    print("All required packages are already available.")


If you need a local install in this notebook, use:
%pip install -U selfies transformers sentencepiece huggingface_hub
All required packages are already available.


In [2]:
from __future__ import annotations

import json
import random
import os
import sys
import time
from pathlib import Path

import selfies
import torch
import transformers

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
assert (PROJECT_ROOT / "src").exists(), "Run this notebook from the Thesis repo root or from notebooks/."

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_collection import BioT5DiverseBeamGenerator
from molecules.collection.filtering import CollectionMetricConfig, prepare_reference_groups
from notebooks.biot5_collection_review_support import (
    assess_biot5_native_generation_output,
    summarize_biot5_native_review_records,
)
from src.io_utils import read_jsonl
from src.prompting import build_text2mol_prompt


def print_section(title: str) -> None:
    print(f"\n=== {title} ===")


def print_json(title: str, payload) -> None:
    print_section(title)
    print(json.dumps(payload, indent=2, ensure_ascii=False))


def print_records(title: str, records, *, limit: int | None = None, keys: list[str] | None = None) -> None:
    payload = list(records)
    total = len(payload)
    if keys is not None:
        payload = [{key: row.get(key) for key in keys} for row in payload]
    shown = payload if limit is None else payload[:limit]
    print_section(f"{title} (showing {len(shown)} of {total})")
    print(json.dumps(shown, indent=2, ensure_ascii=False))


version_report = {
    "project_root": str(PROJECT_ROOT),
    "python": sys.version.split()[0],
    "selfies": selfies.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
}
print_json("Environment", version_report)



=== Environment ===
{
  "project_root": "/home/abril/Documents/HSE/Thesis/Thesis",
  "python": "3.11.14",
  "selfies": "2.1.1",
  "torch": "2.10.0+cu128",
  "transformers": "5.5.0"
}


In [ ]:
TRAIN_FILE = PROJECT_ROOT / "data" / "chebi20" / "processed" / "train.jsonl"
MODEL_NAME_OR_PATH = "QizhiPei/biot5-plus-base-chebi20"
DEVICE = "auto"

SELECTED_DESCRIPTION_COUNT = 2
SELECTION_SEED = 42
NUM_SAMPLES = 30
MODEL_MAX_LENGTH = 512
ALLOW_FILTER_FALLBACK = True

GENERATION_CONFIGS = {
    "greedy_native": {
        "target_count": 1,
        "max_length": 512,
        "num_beams": 1,
        "num_return_sequences": 1,
    },
    "diverse_beam_fast": {
        "target_count": NUM_SAMPLES,
        "max_length": 512,
        "num_beams": 30,
        "num_return_sequences": 30,
        "num_beam_groups": 5,
        "diversity_penalty": 0.3,
        "early_stopping": True,
        "length_penalty": 1.0,
    },
    "diverse_beam_alt": {
        "target_count": NUM_SAMPLES,
        "max_length": 512,
        "num_beams": 30,
        "num_return_sequences": 30,
        "num_beam_groups": 6,
        "diversity_penalty": 0.5,
        "early_stopping": True,
        "length_penalty": 1.0,
    },
}

METRIC_CONFIG = CollectionMetricConfig(
    fingerprint_radius=2,
    fingerprint_num_bits=2048,
    acceptance_dice_threshold=0.7,
)

config_preview = {
    "train_file": str(TRAIN_FILE),
    "model_name_or_path": MODEL_NAME_OR_PATH,
    "device": DEVICE,
    "selected_description_count": SELECTED_DESCRIPTION_COUNT,
    "selection_seed": SELECTION_SEED,
    "num_samples": NUM_SAMPLES,
    "model_max_length": MODEL_MAX_LENGTH,
    "allow_filter_fallback": ALLOW_FILTER_FALLBACK,
    "generation_configs": GENERATION_CONFIGS,
}
print_json("Notebook Config", config_preview)



=== Notebook Config ===
{
  "train_file": "/home/abril/Documents/HSE/Thesis/Thesis/data/chebi20/processed/train.jsonl",
  "model_name_or_path": "QizhiPei/biot5-plus-base-chebi20",
  "device": "auto",
  "selected_description_count": 4,
  "selection_seed": 42,
  "num_samples": 30,
  "model_max_length": 512,
  "allow_filter_fallback": true,
  "generation_configs": {
    "greedy_native": {
      "target_count": 1,
      "max_length": 512,
      "num_beams": 1,
      "num_return_sequences": 1
    },
    "diverse_beam_fast": {
      "target_count": 30,
      "max_length": 512,
      "num_beams": 30,
      "num_return_sequences": 30,
      "num_beam_groups": 5,
      "diversity_penalty": 0.3,
      "early_stopping": true,
      "length_penalty": 1.0
    },
    "diverse_beam_alt": {
      "target_count": 30,
      "max_length": 512,
      "num_beams": 30,
      "num_return_sequences": 30,
      "num_beam_groups": 6,
      "diversity_penalty": 0.5,
      "early_stopping": true,
      "length_p

In [4]:
all_train_records = read_jsonl(TRAIN_FILE)
records_by_id = {
    str(record.get("id")): record
    for record in all_train_records
    if record.get("id") is not None
}
available_description_ids = list(records_by_id)
if len(available_description_ids) < SELECTED_DESCRIPTION_COUNT:
    raise ValueError(
        f"Requested {SELECTED_DESCRIPTION_COUNT} description IDs, but only {len(available_description_ids)} are available."
    )

selection_rng = random.Random(SELECTION_SEED)
SELECTED_DESCRIPTION_IDS = selection_rng.sample(available_description_ids, k=SELECTED_DESCRIPTION_COUNT)
selected_records = [records_by_id[record_id] for record_id in SELECTED_DESCRIPTION_IDS]
print_json(
    "Selected ID Report",
    {
        "num_available_description_ids": len(available_description_ids),
        "selected_description_count": len(SELECTED_DESCRIPTION_IDS),
        "selection_seed": SELECTION_SEED,
        "selected_description_ids_preview": SELECTED_DESCRIPTION_IDS[:20],
    },
)

reference_groups = prepare_reference_groups(all_train_records, METRIC_CONFIG)

selected_preview = [
    {
        "id": str(record.get("id")),
        "description": str(record.get("description")),
        "reference_selfies": str(record.get("selfies")),
        "reference_smiles": str(record.get("source_smiles")),
    }
    for record in selected_records
]
print_records("Selected descriptions", selected_preview)



=== Selected ID Report ===
{
  "num_available_description_ids": 26402,
  "selected_description_count": 4,
  "selection_seed": 42,
  "selected_description_ids_preview": [
    "127693",
    "11135653",
    "7016562",
    "5280493"
  ]
}


[03:38:53] WARNING: not removing hydrogen atom without neighbors
[03:38:55] WARNING: not removing hydrogen atom without neighbors
[03:38:55] WARNING: not removing hydrogen atom without neighbors
[03:38:56] WARNING: not removing hydrogen atom without neighbors
[03:38:56] WARNING: not removing hydrogen atom without neighbors
[03:38:59] WARNING: not removing hydrogen atom without neighbors
[03:38:59] WARNING: not removing hydrogen atom without neighbors
[03:39:00] WARNING: not removing hydrogen atom without neighbors
[03:39:00] WARNING: not removing hydrogen atom without neighbors



=== Selected descriptions (showing 4 of 4) ===
[
  {
    "id": "127693",
    "description": "The molecule is a tetracenomycin that is 1-methyl-11-oxo-6,11-dihydrotetracene-2-carboxylic acid bearing four hydroxy substituents at positions 3, 8, 10 and 12. It is a tetracenomycin, a hydroxy monocarboxylic acid and a member of phenols. It is a conjugate acid of a tetracenomycin F1(1-).",
    "reference_selfies": "[C][C][=C][Branch1][=Branch1][C][=Branch1][C][=O][O][C][Branch1][C][O][=C][C][=C][C][=C][Branch1][#Branch2][C][Branch1][C][O][=C][Ring1][#C][Ring1][#Branch1][C][=Branch1][C][=O][C][=C][Branch1][C][O][C][=C][Branch1][C][O][C][=C][Ring1][Branch2][C][Ring1][S]",
    "reference_smiles": "Cc1c(C(=O)O)c(O)cc2cc3c(c(O)c12)C(=O)c1c(O)cc(O)cc1C3"
  },
  {
    "id": "11135653",
    "description": "The molecule is a fatty acyl-AMP that results from the formal condensation of the phosphoryl group of AMP with the carboxyl group of hexadecanoic (palmitic) acid. It derives from a hexadecanoic ac

[03:39:00] WARNING: not removing hydrogen atom without neighbors


In [5]:
prompt_variants_by_id = {}
for record in selected_records:
    description_id = str(record.get("id"))
    description = str(record.get("description"))
    prompt_variants_by_id[description_id] = {
        "native_selfies": build_text2mol_prompt(description),
    }

prompt_preview = []
for record in selected_records:
    description_id = str(record.get("id"))
    for prompt_variant, prompt_text in prompt_variants_by_id[description_id].items():
        prompt_preview.append(
            {
                "description_id": description_id,
                "prompt_variant": prompt_variant,
                "prompt_text": prompt_text,
            }
        )

print_records("Prompt preview", prompt_preview)



=== Prompt preview (showing 4 of 4) ===
[
  {
    "description_id": "127693",
    "prompt_variant": "native_selfies",
    "prompt_text": "Definition: You are given a molecule description in English. Your job is to generate the molecule SELFIES that fits the description.\n\nNow complete the following example -\nInput: The molecule is a tetracenomycin that is 1-methyl-11-oxo-6,11-dihydrotetracene-2-carboxylic acid bearing four hydroxy substituents at positions 3, 8, 10 and 12. It is a tetracenomycin, a hydroxy monocarboxylic acid and a member of phenols. It is a conjugate acid of a tetracenomycin F1(1-).\nOutput: "
  },
  {
    "description_id": "11135653",
    "prompt_variant": "native_selfies",
    "prompt_text": "Definition: You are given a molecule description in English. Your job is to generate the molecule SELFIES that fits the description.\n\nNow complete the following example -\nInput: The molecule is a fatty acyl-AMP that results from the formal condensation of the phosphoryl g

In [6]:
# Use the shared active collection generator so notebook review matches collection-time inference.


In [7]:
from huggingface_hub import login

access_token = os.environ.get("HF_TOKEN")
if access_token:
    login(token=access_token)

first_generation_config_name = next(iter(GENERATION_CONFIGS))
generator = BioT5DiverseBeamGenerator(
    model_name_or_path=MODEL_NAME_OR_PATH,
    device_name=DEVICE,
    model_max_length=MODEL_MAX_LENGTH,
    generation_config=GENERATION_CONFIGS[first_generation_config_name],
)

generator_preview = {
    "device": str(generator.device),
    "supports_remote_group_beam_search": bool(generator.supports_remote_group_beam_search),
    "num_selected_descriptions": len(selected_records),
    "generation_config_names": list(GENERATION_CONFIGS.keys()),
}
print_json("Generator Preview", generator_preview)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



=== Generator Preview ===
{
  "device": "cuda",
  "supports_remote_group_beam_search": true,
  "num_selected_descriptions": 4,
  "generation_config_names": [
    "greedy_native",
    "diverse_beam_fast",
    "diverse_beam_alt"
  ]
}


In [8]:
raw_generations = []
for generation_config_name, generation_config in GENERATION_CONFIGS.items():
    generator.generation_config = dict(generation_config)
    target_count = int(generation_config.get("target_count", NUM_SAMPLES))
    print_section(f"Running {generation_config_name}")

    for record in selected_records:
        description_id = str(record.get("id"))
        description = str(record.get("description"))
        for prompt_variant, prompt_text in prompt_variants_by_id[description_id].items():
            started_at = time.perf_counter()
            outputs = generator.generate_candidates(prompt_text, target_count)
            elapsed_seconds_batch = time.perf_counter() - started_at
            seconds_per_sample_batch = elapsed_seconds_batch / max(len(outputs), 1)
            print(
                f"{generation_config_name} | {description_id} | {prompt_variant} | "
                f"samples={len(outputs)} | elapsed_seconds={elapsed_seconds_batch:.2f} | "
                f"seconds_per_sample={seconds_per_sample_batch:.3f}"
            )
            for candidate_index, raw_prediction_text in enumerate(outputs):
                raw_generations.append(
                    {
                        "generation_config_name": generation_config_name,
                        "description_id": description_id,
                        "description": description,
                        "prompt_variant": prompt_variant,
                        "candidate_index": candidate_index,
                        "raw_prediction_text": raw_prediction_text,
                        "elapsed_seconds_batch": elapsed_seconds_batch,
                        "seconds_per_sample_batch": seconds_per_sample_batch,
                    }
                )

generation_report = {
    "num_rows": len(raw_generations),
    "generation_configs": sorted({row["generation_config_name"] for row in raw_generations}),
    "description_ids": sorted({row["description_id"] for row in raw_generations}),
    "prompt_variants": sorted({row["prompt_variant"] for row in raw_generations}),
}
print_json("Generation Report", generation_report)
print_records(
    "Raw generation preview",
    raw_generations,
    limit=12,
    keys=[
        "generation_config_name",
        "description_id",
        "prompt_variant",
        "candidate_index",
        "raw_prediction_text",
    ],
)



=== Running greedy_native ===
greedy_native | 127693 | native_selfies | samples=1 | elapsed_seconds=0.99 | seconds_per_sample=0.987
greedy_native | 11135653 | native_selfies | samples=1 | elapsed_seconds=0.70 | seconds_per_sample=0.701
greedy_native | 7016562 | native_selfies | samples=1 | elapsed_seconds=0.23 | seconds_per_sample=0.234
greedy_native | 5280493 | native_selfies | samples=1 | elapsed_seconds=0.77 | seconds_per_sample=0.774

=== Running diverse_beam_fast ===
diverse_beam_fast | 127693 | native_selfies | samples=30 | elapsed_seconds=5.66 | seconds_per_sample=0.189
diverse_beam_fast | 11135653 | native_selfies | samples=30 | elapsed_seconds=3.83 | seconds_per_sample=0.128
diverse_beam_fast | 7016562 | native_selfies | samples=30 | elapsed_seconds=1.53 | seconds_per_sample=0.051
diverse_beam_fast | 5280493 | native_selfies | samples=30 | elapsed_seconds=3.28 | seconds_per_sample=0.109

=== Running diverse_beam_alt ===


OutOfMemoryError: CUDA out of memory. Tried to allocate 92.00 MiB. GPU 0 has a total capacity of 5.68 GiB of which 17.12 MiB is free. Including non-PyTorch memory, this process has 5.54 GiB memory in use. Of the allocated memory 3.53 GiB is allocated by PyTorch, and 1.88 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
review_rows = []
for row in raw_generations:
    review_rows.append(
        assess_biot5_native_generation_output(
            description_id=row["description_id"],
            description=row["description"],
            prompt_variant=row["prompt_variant"],
            candidate_index=row["candidate_index"],
            raw_prediction_text=row["raw_prediction_text"],
            references=reference_groups[row["description"]],
            metric_config=METRIC_CONFIG,
            generation_config_name=row["generation_config_name"],
            elapsed_seconds_batch=row["elapsed_seconds_batch"],
            seconds_per_sample_batch=row["seconds_per_sample_batch"],
            allow_filter_fallback=ALLOW_FILTER_FALLBACK,
        )
    )

parsed_preview_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "candidate_index",
    "raw_prediction_text",
    "cleaned_selfies",
    "parsed_selfies",
    "filtered_selfies",
    "used_filter_selfies_fallback",
    "decoded_smiles",
    "selfies_decode_error",
]
print_records("Raw vs Parsed SELFIES preview", review_rows, limit=20, keys=parsed_preview_columns)

detail_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "candidate_index",
    "raw_prediction_text",
    "cleaned_selfies",
    "parsed_selfies",
    "selected_selfies",
    "filtered_selfies",
    "used_filter_selfies_fallback",
    "decoded_smiles",
    "is_valid_selfies",
    "selfies_decode_error",
    "canonical_smiles",
    "derived_selfies",
    "is_valid_smiles",
    "best_reference_smiles",
    "max_dice_similarity",
    "passes_similarity_threshold",
    "rejection_reason",
]
print_records("Assessment preview", review_rows, limit=30, keys=detail_columns)


In [ ]:
summary_rows = summarize_biot5_native_review_records(review_rows)
summary_rows = sorted(
    summary_rows,
    key=lambda item: (
        str(item["generation_config_name"]),
        str(item["description_id"]),
        str(item["prompt_variant"]),
    ),
)

summary_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "sample_count",
    "elapsed_seconds_batch",
    "seconds_per_sample_batch",
    "valid_selfies_rate",
    "filter_selfies_fallback_rate",
    "filter_selfies_recovery_rate",
    "valid_smiles_rate",
    "unique_canonical_smiles_count",
    "avg_max_dice_similarity",
    "best_max_dice_similarity",
    "passes_similarity_threshold_rate",
    "invalid_selfies_rate",
    "invalid_smiles_rate",
]
print_records("Summary rows", summary_rows, keys=summary_columns)

print_section("Compact Summary")
for summary in summary_rows:
    print(
        f"{summary['generation_config_name']} | {summary['description_id']} | {summary['prompt_variant']} | "
        f"samples={summary['sample_count']} | elapsed={summary['elapsed_seconds_batch']:.2f}s | "
        f"sec_per_sample={summary['seconds_per_sample_batch']:.3f} | "
        f"valid_selfies={summary['valid_selfies_rate']:.3f} | "
        f"filter_recovery={summary['filter_selfies_recovery_rate']:.3f} | "
        f"valid_smiles={summary['valid_smiles_rate']:.3f} | "
        f"unique={summary['unique_canonical_smiles_count']} | "
        f"avg_dice={summary['avg_max_dice_similarity']:.3f} | "
        f"best_dice={summary['best_max_dice_similarity']:.3f}"
    )


## Review Questions

After the notebook runs, use the summary and assessment preview to answer:

1. Does the native BioT5 path produce decodable SELFIES more reliably than the old custom tokenizer path?
2. How often is `filter_selfies(...)` needed to recover otherwise valid outputs?
3. Does diverse beam improve usable molecule diversity without collapsing validity?
4. Is the native checkpoint strong enough that the old SMILES-tag extraction experiment is no longer needed for this review notebook?
